## The code here is for training CapHLA_el model.
## Users only need to change the corresponding trainging dataset to the same format in order to retrain the model based on their own data

In [11]:
import pandas as pd
from io import StringIO
from torch.utils import data
from torch import nn
import torch
import math
from torch.nn import functional as F
import pandas as pd
import warnings
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score, auc
from sklearn.metrics import precision_recall_curve

In [3]:
# one hot encoder MHC and peptide sequence
aa_dict_tokens = {'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6,
                  'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13,
                  'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20}
aa_dict_one_hot = {}
for i in aa_dict_tokens.keys():
    aa_dict_one_hot[i] = [
        1 if j == aa_dict_tokens[i] else 0 for j in range(len(aa_dict_tokens))]


def read_file(file_path, max_len=25):
    df = pd.read_csv(file_path, header=0)
    df['padding'] = df['peptide'].apply(
        lambda x: x + 'X' * (max_len - len(x)))
    data_MHC = [list(i) for i in df['MHC pseudo-seq'].tolist()]
    data_pep = [list(i) for i in df['padding'].tolist()]
    data_label = df['label'].tolist()
    return [data_pep, data_MHC, data_label]


class Pep_MHC_dataset(data.Dataset):
    def __init__(self, dataset, aa_dict, vocab=None):
        self.pep = torch.tensor(
            [[aa_dict[i] for i in line] for line in dataset[0]])
        self.hla = torch.tensor(
            [[aa_dict[i] for i in line] for line in dataset[1]])
        self.label = torch.tensor(dataset[2])

    def __getitem__(self, idx):
        return (self.pep[idx], self.hla[idx]), self.label[idx]

    def __len__(self):
        return len(self.pep)


def load_data(file_path, batch_size, aa_dict):
    data_all = read_file(file_path)
    dataset = Pep_MHC_dataset(data_all, aa_dict)
    data_iter = data.DataLoader(dataset, batch_size,
                                shuffle=False, num_workers=8)
    return data_iter


def load_data_pd(df, batch_size, aa_dict, max_len=25):
    df['padding'] = df['peptide'].apply(
        lambda x: x + 'X' * (max_len - len(x)))
    data_MHC = [list(i) for i in df['MHC pseudo-seq'].tolist()]
    data_pep = [list(i) for i in df['padding'].tolist()]
    data_label = df['label'].tolist()
    dataset = Pep_MHC_dataset([data_pep, data_MHC, data_label], aa_dict)
    data_iter = data.DataLoader(dataset, batch_size,
                                shuffle=False, num_workers=8)
    return data_iter

In [4]:
r"""
Args:
    input (torch.Tensor): with shape `(Batch_size, num_step, vocab_size)`.
    queries, keys, values all equal to input

Returns:
    torch.Tensor: output, with shape `(Batch_size, num_step, vocab_size)`.
"""


class DotProductionAttention(nn.Module):
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values):
        d = queries.shape[-1]
        score = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention = nn.functional.softmax(score, dim=-1)
        return torch.bmm(self.dropout(self.attention), values)


def transpose_qkv(X, num_heads):
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class MultiHeadAttention(nn.Module):
    def __init__(self, vocab_size, num_heads,
                 dropout, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductionAttention(dropout)
        self.wq = nn.Linear(vocab_size, vocab_size*num_heads, bias=bias)
        self.kw = nn.Linear(vocab_size, vocab_size*num_heads, bias=bias)
        self.wv = nn.Linear(vocab_size, vocab_size*num_heads, bias=bias)
        self.wo = nn.Linear(vocab_size*num_heads, vocab_size,  bias=bias)

    def forward(self, queries, keys, values):
        queries = transpose_qkv(self.wq(queries), self.num_heads)
        keys = transpose_qkv(self.kw(keys), self.num_heads)
        values = transpose_qkv(self.wv(values), self.num_heads)
        output = self.attention(queries, keys, values)
        output_concat = transpose_output(output, self.num_heads)
        return self.wo(output_concat)

In [5]:
r"""
Args:
    input (torch.Tensor): with shape `(Batch_size, num_step, vocab_size)`.
    queries, keys, values all equal to input

Returns:
    torch.Tensor: output, with shape `(Batch_size, num_step, vocab_size)`.
"""


class ConvolutionModule(nn.Module):
    def __init__(self, vocab_size, num_channels, depthwise_kernel_size,
                 dropout, bias, use_group_norm):
        super().__init__()
        if (depthwise_kernel_size - 1) % 2 != 0:
            raise ValueError("depthwise_kernel_size must be odd to achieve 'SAME' padding.")
        self.sequential = nn.Sequential(
            nn.Conv1d(
                vocab_size,
                2 * num_channels,
                1,
                stride=1,
                padding=0,
                bias=bias,
            ),
            nn.GLU(dim=1),
            nn.Conv1d(
                num_channels,
                num_channels,
                depthwise_kernel_size,
                stride=1,
                padding=(depthwise_kernel_size - 1) // 2,
                groups=num_channels,
                bias=bias,
            ),
            nn.GroupNorm(num_groups=1, num_channels=num_channels)
            if use_group_norm
            else nn.BatchNorm1d(num_channels),
            torch.nn.SiLU(),
            torch.nn.Conv1d(
                num_channels,
                vocab_size,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=bias,
            ),
            torch.nn.Dropout(dropout),
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        x = input.transpose(1, 2)
        x = self.sequential(x)
        return x.transpose(1, 2)

In [7]:
r'''

convolution model(pointwise | depthwise | pointwise) +
multiself attention

flatten + feature selection (full connection)
'''


class FeatureSelection(torch.nn.Module):
    def __init__(self, input_dim, num_hiddens, dropout=0.2):
        super().__init__()
        self.sequential = torch.nn.Sequential(
            torch.nn.Linear(input_dim, num_hiddens, bias=True),
            torch.nn.SiLU(),
            torch.nn.BatchNorm1d(num_hiddens),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(num_hiddens, 64, bias=True),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 2, bias=True)
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self.sequential(input)


class CapHLA(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_hiddens: int,
        num_heads: int,
        num_step: int,
        num_channels: int,
        depthwise_kernel_size,
        dropout,
        bias=False,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.norm = nn.LayerNorm(vocab_size)
        self.selfattention = MultiHeadAttention(vocab_size, num_heads, dropout)
        self.conv = ConvolutionModule(
            vocab_size=vocab_size,
            num_channels=num_channels,
            depthwise_kernel_size=depthwise_kernel_size,
            dropout=dropout,
            bias=True,
            use_group_norm=False,
        )
        self.flatten = nn.Flatten(start_dim=1, end_dim=-1)
        self.feature_selection = FeatureSelection(
            vocab_size * num_step, num_hiddens)

    def forward(self, pep, mhc):
        X = torch.cat((pep, mhc), dim=1).type(torch.float32)
        residual = X
        X = self.conv(X)
        X = self.norm(residual + X)
        residual = X
        X = self.selfattention(X, X, X)
        X = self.norm(residual + X)
        X = self.flatten(X)
        return self.feature_selection(X)

In [6]:
def evaluate_model(net, test_iter, device):
    net.eval()
    with torch.no_grad():
        label_metric = torch.tensor([]).to(device)
        softmax_metric = torch.tensor([]).to(device)
        y_metric = torch.tensor([]).to(device)
        for X, y in test_iter:
            y = y.to(device)
            y_metric = torch.cat((y_metric, y), dim=0)
            if isinstance(X, list):
                X = [x.to(device) for x in X]
            else:
                X = X.to(device)
            y_hat = net(X[0], X[1])
            y_hat_softmax = F.softmax(y_hat, dim=1)
            softmax_metric = torch.cat((softmax_metric, y_hat_softmax), dim=0)
            y_hat_label = y_hat.argmax(dim=1)
            label_metric = torch.cat((label_metric, y_hat_label), dim=0)
        softmax_metric = softmax_metric[:, 1].cpu().numpy()
        true_label = y_metric.cpu().numpy()
        pred_label = label_metric.cpu().numpy()
        eva = evalution(true_label, pred_label, softmax_metric)
    return eva

In [8]:
def train_model(net, num_epoch, train_iter, val_iter, trainer, loss,
                devices):
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    net.train()
    for epoch in range(num_epoch):
        for i, (X, y) in enumerate(train_iter):
            if isinstance(X, list):
                X = [x.to(devices[0]) for x in X]
            else:
                X.to(devices[0])
            y = y.to(devices[0])
            trainer.zero_grad()
            y_hat = net(X[0], X[1])
            lo = loss(y_hat, y)
            lo.sum().backward()
            trainer.step()

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

In [10]:
def evalution(true_label, pred_label, pred_score=None):
    tn, fp, fn, tp = confusion_matrix(true_label, pred_label).ravel()
    accuracy = (tp + tn) / (tn + fp + fn + tp)
#    mcc = ((tp * tn) - (fn * fp)) / np.sqrt((tp + fn) * (tn + fp) * (tp + fp) * (tn + fn))
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = (2 * precision * recall) / (precision + recall)
    if pred_score is None:
        roc_auc, aupr = np.nan, np.nan
    else:
        roc_auc = roc_auc_score(true_label, pred_score)
        prec, reca, _ = precision_recall_curve(true_label, pred_score)
        aupr = auc(reca, prec)
    return [roc_auc, accuracy, f1, sensitivity, specificity, precision, recall, aupr]

In [14]:
num_hiddens, num_heads, dropout = 800, 9, 0.2
num_channels, depthwise_kernel_size = 3200, 9
num_step = 59
vocab_size = 21
num_epoch = 50

model_result = []
os.makedirs('el_params/', exist_ok=True)
for i in tqdm(range(5)):
    train_path = f'el_dataset/train_fold{i}.csv'
    val_path = f'el_dataset/validation_fold{i}.csv'
    ex_path = f'el_dataset/external.csv'
    batch_size = 1024
    train_iter = load_data(train_path, batch_size, aa_dict_one_hot)
    val_iter = load_data(val_path, batch_size, aa_dict_one_hot)
    ex_iter = load_data(ex_path, batch_size, aa_dict_one_hot)

    net = CapHLA(vocab_size, num_hiddens, num_heads,
                 num_step, num_channels,
                 depthwise_kernel_size, dropout, bias=False)
    net.apply(init_weights)
    loss = nn.CrossEntropyLoss(reduction='none')
    trainer = torch.optim.Adam(net.parameters(), lr=0.001)
    devices = [torch.device('cuda:0'), torch.device('cuda:1')]
    train_model(net, num_epoch, train_iter, val_iter, trainer, loss, devices)

    path = f'el_params/fold{i}.params'
    torch.save(net.state_dict(), path)

    eva_val = evaluate_model(net, val_iter, devices[0])
    eva_val = ['validation', f'fold{i}'] + eva_val
    model_result.append(eva_val)
    eva_ex = evaluate_model(net, ex_iter, devices[0])
    eva_ex = ['external', f'fold{i}'] + eva_ex
    model_result.append(eva_ex)

model_result = pd.DataFrame(model_result, columns=[
    'type', 'fold', 'roc_auc', 'accuracy', 'F1', 'sensitivity',
    'specificity', 'precision', 'recall', 'aupr'])
model_result.to_csv('el_params/CapHLA.csv', index=False)

100%|█████████████████████████████████████████| 5/5 [6:52:14<00:00, 4946.84s/it]
